In [9]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')
import pandas as pd
import numpy as np
from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

In [10]:
s = S3Client()
files = s.list_files('live/candles/1second/USDINR26SEPFUT/')
dates = sorted([f.split('/')[-1].replace('.parquet','') for f in files])

db = DuckDBClient()
all_df = []
for date in dates:
    df = db.read_parquet(f'live/candles/1second/USDINR26SEPFUT/{date}.parquet')
    df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
    df = df[(df['ist_sec'] >= 9*3600) & (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
    df['date'] = date
    all_df.append(df)
db.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
df = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
df['mid']     = (df['bid_p1'] + df['ask_p1']) / 2
df['spread']  = df['ask_p1'] - df['bid_p1']
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
df = df[df['spread'] > 0].copy()
print(f'Total bars: {len(df)} across {len(dates)} days: {dates}')
print()

Total bars: 121654 across 7 days: ['2026-09-03', '2026-09-04', '2026-09-17', '2026-09-18', '2026-09-19', '2026-09-20', '2026-09-21']



In [14]:
print('=== Per-Day Market Statistics ===')
print(f'{"Date":>12} {"med_spread":>12} {"mean_spread":>12} {"sigma_log":>12} {"sigma_price":>14} {"bars":>7}')
print('-' * 75)
kappa = 1.3282
day_stats = []
for date in dates:
    mask      = df['date'] == date
    med_sp    = df.loc[mask, 'spread'].median()
    mean_sp   = df.loc[mask, 'spread'].mean()
    sig_log   = df.loc[mask, 'log_ret'].std()
    sig_price = sig_log * df.loc[mask, 'close'].mean()
    n         = mask.sum()
    day_stats.append({
        'date': date, 'med_spread': med_sp,
        'sigma_log': sig_log, 'sigma_price': sig_price
    })
    print(f'{date:>12} {med_sp:>12.5f} {mean_sp:>12.5f} {sig_log:>12.6f} {sig_price:>14.6f} {n:>7}')

overall_med    = df['spread'].median()
overall_sigma  = df['log_ret'].std()
overall_sigmap = overall_sigma * df['close'].mean()
print(f'{"OVERALL":>12} {overall_med:>12.5f} {"—":>12} {overall_sigma:>12.6f} {overall_sigmap:>14.6f} {len(df):>7}')

=== Per-Day Market Statistics ===
        Date   med_spread  mean_spread    sigma_log    sigma_price    bars
---------------------------------------------------------------------------
  2026-09-03      0.00750      0.00860     0.000038       0.003557   43769
  2026-09-04      0.00500      0.00721     0.000016       0.001503   23755
  2026-09-17      0.00750      0.00940     0.000072       0.006927   54130
  2026-09-18          nan          nan          nan            nan       0
  2026-09-19          nan          nan          nan            nan       0
  2026-09-20          nan          nan          nan            nan       0
  2026-09-21          nan          nan          nan            nan       0
     OVERALL      0.00750            —     0.000054       0.005105  121654


In [16]:
print()
print('=== Gamma Calibration — Fine Grid (all days combined) ===')
print(f'Target spread (median): {overall_med:.5f} paise')
print()
print(f'{"gamma":>8} {"base_spread":>12} {"diff":>10} {"pct_diff":>10}')
print('-' * 45)
print()
print('=== Gamma Calibration — Fine Grid (all days combined) ===')
print(f'Target spread (median): {overall_med:.5f} paise')
print()
print(f'{"gamma":>8} {"base_spread":>12} {"diff":>10} {"pct_diff":>10}')
print('-' * 45)

gammas = list(range(50, 301, 10)) + list(range(300, 601, 25))
best_gamma = None
best_diff  = float('inf')

for gamma in gammas:
    base_spread = (2/kappa) * np.log(1 + kappa/gamma)
    diff        = base_spread - overall_med
    pct_diff    = diff / overall_med * 100
    marker      = ' ← best' if abs(diff) < best_diff else ''
    if abs(diff) < best_diff:
        best_diff  = abs(diff)
        best_gamma = gamma
    if abs(pct_diff) < 20:  # only print near-target values
        print(f'{gamma:>8} {base_spread:>12.5f} {diff:>10.5f} {pct_diff:>9.1f}%{marker}')

print()
print(f'Best gamma: {best_gamma}')
print(f'Spread at best gamma: {(2/kappa)*np.log(1+kappa/best_gamma):.5f}')
print(f'Target spread:        {overall_med:.5f}')


=== Gamma Calibration — Fine Grid (all days combined) ===
Target spread (median): 0.00750 paise

   gamma  base_spread       diff   pct_diff
---------------------------------------------

=== Gamma Calibration — Fine Grid (all days combined) ===
Target spread (median): 0.00750 paise

   gamma  base_spread       diff   pct_diff
---------------------------------------------
     230      0.00867    0.00117      15.6% ← best
     240      0.00831    0.00081      10.8% ← best
     250      0.00798    0.00048       6.4% ← best
     260      0.00767    0.00017       2.3% ← best
     270      0.00739   -0.00011      -1.5% ← best
     280      0.00713   -0.00037      -5.0%
     290      0.00688   -0.00062      -8.3%
     300      0.00665   -0.00085     -11.3%
     300      0.00665   -0.00085     -11.3%
     325      0.00614   -0.00136     -18.1%

Best gamma: 270
Spread at best gamma: 0.00739
Target spread:        0.00750


In [17]:
print()
print('=== Per-Day Optimal Gamma ===')
print(f'{"Date":>12} {"target_spread":>14} {"opt_gamma":>10} {"achieved_spread":>16}')
print('-' * 56)

for stat in day_stats:
    target = stat['med_spread']
    opt_g  = None
    opt_d  = float('inf')
    for gamma in range(50, 2001, 5):
        spread = (2/kappa) * np.log(1 + kappa/gamma)
        if abs(spread - target) < opt_d:
            opt_d = abs(spread - target)
            opt_g = gamma
    achieved = (2/kappa) * np.log(1 + kappa/opt_g)
    print(f'{stat["date"]:>12} {target:>14.5f} {opt_g:>10} {achieved:>16.5f}')

# ── Sigma stability ────────────────────────────────────────────────
print()
print('=== Sigma by Hour (all days pooled) ===')
df['hour'] = df['ist_sec'] // 3600
print(f'{"Hour":>6} {"sigma_log":>12} {"sigma_price":>14} {"med_spread":>12} {"n_bars":>8}')
print('-' * 55)

for hour in range(9, 17):
    mask      = df['hour'] == hour
    sig_log   = df.loc[mask, 'log_ret'].std()
    sig_price = sig_log * df.loc[mask, 'close'].mean()
    med_sp    = df.loc[mask, 'spread'].median()
    n         = mask.sum()
    print(f'{hour:>4}:00 {sig_log:>12.6f} {sig_price:>14.6f} {med_sp:>12.5f} {n:>8}')



=== Per-Day Optimal Gamma ===
        Date  target_spread  opt_gamma  achieved_spread
--------------------------------------------------------
  2026-09-03        0.00750        265          0.00753
  2026-09-04        0.00500        400          0.00499
  2026-09-17        0.00750        265          0.00753


TypeError: unsupported operand type(s) for /: 'float' and 'NoneType'

In [18]:
# Check what's actually in the parquet for Sep 18
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')
from data.store.duckdb_client import DuckDBClient

db = DuckDBClient()
df = db.read_parquet('live/candles/1second/USDINR26SEPFUT/2026-09-18.parquet')
print(f'Sep 18 total rows: {len(df)}')
print(f'Columns: {df.columns.tolist()[:5]}')
if len(df) > 0:
    df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
    print(f'IST hours: {sorted(df["ist_sec"].apply(lambda x: x//3600).unique())}')
    mkt = df[(df['ist_sec'] >= 9*3600) & (df['ist_sec'] <= 17*3600)]
    print(f'Market hours bars: {len(mkt)}')
db.close()

Sep 18 total rows: 92918
Columns: ['symbol', 'ts_sec', 'ts_ist', 'open', 'high']
IST hours: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
Market hours bars: 31843


In [20]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import pandas as pd
import numpy as np
from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

s = S3Client()
files = s.list_files('live/candles/1second/USDINR26SEPFUT/')
dates = sorted([f.split('/')[-1].replace('.parquet','') for f in files])

db = DuckDBClient()
all_df = []
for date in dates:
    df = db.read_parquet(f'live/candles/1second/USDINR26SEPFUT/{date}.parquet')
    df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
    # Market hours filter FIRST — before anything else
    df = df[(df['ist_sec'] >= 9*3600) & (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
    if len(df) < 100:
        print(f'{date}: skipped ({len(df)} bars)')
        continue
    df['date'] = date
    all_df.append(df)
    print(f'{date}: {len(df)} bars loaded')
db.close()

df = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
df['mid']    = (df['bid_p1'] + df['ask_p1']) / 2
df['spread'] = df['ask_p1'] - df['bid_p1']
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

# Remove crossed quotes only
df = df[df['spread'] > 0].copy()
print(f'\nTotal clean bars: {len(df)} across {df["date"].nunique()} days')

2026-09-03: 43770 bars loaded
2026-09-04: 23755 bars loaded


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-09-17: 54131 bars loaded
2026-09-18: 31843 bars loaded
2026-09-19: 32263 bars loaded
2026-09-20: 32281 bars loaded
2026-09-21: 32663 bars loaded

Total clean bars: 121654 across 3 days


In [22]:
db = DuckDBClient()
df21 = db.read_parquet('live/candles/1second/USDINR26SEPFUT/2026-09-21.parquet')
db.close()

df21['ist_sec'] = (df21['ts_sec'] + 19800) % 86400
df21 = df21[(df21['ist_sec'] >= 9*3600) & (df21['ist_sec'] <= 17*3600)].reset_index(drop=True)

print(f'Sep 21 market hours bars: {len(df21)}')
print(f'\nNon-zero columns:')
for col in df21.columns:
    non_zero = (df21[col] != 0).sum()
    if non_zero > 0:
        print(f'  {col}: {non_zero} non-zero values')

Sep 21 market hours bars: 32663

Non-zero columns:
  symbol: 32663 non-zero values
  ts_sec: 32663 non-zero values
  ts_ist: 32663 non-zero values
  open: 32663 non-zero values
  high: 32663 non-zero values
  low: 32663 non-zero values
  close: 32663 non-zero values
  oi: 32663 non-zero values
  tick_count: 32663 non-zero values
  imbalance_mean: 32663 non-zero values
  spread_mean: 32663 non-zero values
  ist_sec: 32663 non-zero values


In [24]:
print(df21[['ts_ist','open','high','low','close','imbalance_mean','spread_mean','oi']].head(10))
print()
print(f'spread_mean stats:')
print(df21['spread_mean'].describe())
print()
print(f'imbalance_mean stats:')
print(df21['imbalance_mean'].describe())

                ts_ist     open   high     low    close  imbalance_mean  \
0  2026-09-21 03:30:00  95.9375  95.95  95.925  95.9375       -0.625913   
1  2026-09-21 03:30:02  95.9375  95.95  95.925  95.9375       -0.625913   
2  2026-09-21 03:30:03  95.9375  95.95  95.925  95.9375       -0.625913   
3  2026-09-21 03:30:05  95.9375  95.95  95.925  95.9375       -0.625913   
4  2026-09-21 03:30:06  95.9375  95.95  95.925  95.9375       -0.625913   
5  2026-09-21 03:30:08  95.9375  95.95  95.925  95.9375       -0.625913   
6  2026-09-21 03:30:10  95.9375  95.95  95.925  95.9375       -0.625913   
7  2026-09-21 03:30:11  95.9375  95.95  95.925  95.9375       -0.625913   
8  2026-09-21 03:30:13  95.9375  95.95  95.925  95.9375       -0.625913   
9  2026-09-21 03:30:14  95.9375  95.95  95.925  95.9375       -0.625913   

   spread_mean         oi  
0      0.02725  2935109.0  
1      0.02725  2935109.0  
2      0.02725  2935109.0  
3      0.02725  2935109.0  
4      0.02725  2935109.0  
5     